# 📘 Notebook 11: Beyond the Basics: Robust ML

### Course: *From Python to Deep Learning & Fine-Tuning*
**Instructor:** Ioannis Tsioulis

*Module C: Machine Learning · Notebook 4 of 4 in this module · (11 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Improve inputs through **feature engineering** and **feature scaling**
- Estimate performance reliably with **cross-validation**
- Combine models into stronger **ensembles** (Random Forest)
- Tune models systematically with **grid search**
- Discover structure without labels: **k-means clustering** and **PCA**

> **Prerequisites:** Notebooks 08-10.
>
> This notebook collects the techniques that separate a toy model from a trustworthy one. They apply just as much to deep learning, so the habits you build here carry forward.

> ℹ️ **Setup note.** If needed: `import piplite; await piplite.install(['scikit-learn','numpy','matplotlib'])`

## 1. Feature scaling

### Why it is necessary
Many algorithms (k-NN, logistic regression, and all neural networks) are sensitive to the **scale** of features. If ‘age’ ranges 0-100 and ‘income’ ranges 0-100000, the income feature dominates distance and gradient computations purely because of its larger numbers. **Scaling** puts all features on a comparable footing.

The most common method, **standardisation**, rescales each feature to mean 0 and standard deviation 1:

$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)
# Two features on wildly different scales
age = np.random.randint(18, 70, 200)
income = np.random.randint(10000, 100000, 200)
X = np.column_stack([age, income]).astype(float)
y = (income > 50000).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# CRITICAL: fit the scaler on TRAINING data only, then apply to both.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn mu, sigma from train
X_test_scaled  = scaler.transform(X_test)         # apply the SAME transform

print("Before scaling (first row):", X_train[0])
print("After scaling  (first row):", X_train_scaled[0].round(3))

> ⚠️ **Common pitfalls**
>
> - **Data leakage through scaling.** Fitting the scaler on the *whole* dataset (before splitting) leaks test-set statistics into training. Always `fit_transform` on the training set and only `transform` the test set, exactly as above.

## 2. Cross-validation

### The problem with a single split
A single train/test split gives one performance estimate, but you might have been lucky or unlucky with which points landed in the test set. **k-fold cross-validation** removes that luck: it splits the data into *k* parts (folds), trains on *k−1* of them and tests on the remaining one, rotating through all folds, then averages the results.

With 5 folds, every point is used for testing exactly once, giving a far more stable and trustworthy estimate.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model = LogisticRegression()
scores = cross_val_score(model, X_train_scaled, y_train, cv=5)

print("Accuracy on each of the 5 folds:", scores.round(3))
print(f"Mean accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

> 🧠 Report the **mean and spread** across folds, not a single number. The spread tells you how stable the model is, a high variance across folds is itself a warning sign.

## 3. Ensemble methods: wisdom of crowds

### Intuition first
A single decision tree overfits easily. But if we train **many** trees, each on a slightly different random sample of the data and features, and let them **vote**, their individual errors tend to cancel out. This is a **Random Forest**, one of the most reliable off-the-shelf classifiers, and a perfect illustration of *ensemble learning*: many weak models combine into a strong one.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

single_tree = DecisionTreeClassifier(random_state=42)
forest = RandomForestClassifier(n_estimators=100, random_state=42)

for name, clf in [("Single tree", single_tree), ("Random forest (100 trees)", forest)]:
    s = cross_val_score(clf, X_train_scaled, y_train, cv=5)
    print(f"{name:28} mean accuracy: {s.mean():.3f}")

### A bonus: feature importance
Random forests can report how useful each feature was, a simple, valuable form of interpretability:

In [ ]:
forest.fit(X_train_scaled, y_train)
for feature, importance in zip(["age", "income"], forest.feature_importances_):
    print(f"{feature:8} importance: {importance:.3f}")

## 4. Systematic tuning with grid search

Models have **hyperparameters**, settings we choose, not learn (e.g. the number of neighbours in k-NN, the depth of a tree). Rather than guess, **grid search** tries every combination from a list and uses cross-validation to pick the best, automatically.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, None],
}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)
grid.fit(X_train_scaled, y_train)

print("Best parameters:", grid.best_params_)
print(f"Best cross-val accuracy: {grid.best_score_:.3f}")

## 5. Unsupervised learning I: k-means clustering

### A different kind of problem
So far every example had labels. **Unsupervised learning** works with *unlabelled* data, finding structure on its own. **k-means clustering** groups points into *k* clusters by repeatedly assigning each point to the nearest cluster centre and then moving each centre to the average of its members.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

Xb, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(Xb)   # note: no y was given!

plt.figure(figsize=(7, 5))
plt.scatter(Xb[:, 0], Xb[:, 1], c=labels, cmap="viridis", alpha=0.6)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c="red", marker="X", s=200, label="centres")
plt.title("k-means clustering (no labels used)")
plt.legend()
plt.show()

## 6. Unsupervised learning II: PCA

### The idea
**Principal Component Analysis (PCA)** is a **dimensionality reduction** technique: it finds new axes (principal components) that capture the most variation in the data, letting us compress many features into a few while losing as little information as possible. It is invaluable for visualising high-dimensional data in 2-D.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = iris.data       # 4 features
y_iris = iris.target

pca = PCA(n_components=2)            # compress 4-D -> 2-D
X_2d = pca.fit_transform(X_iris)

print("Variance kept by 2 components:", pca.explained_variance_ratio_.sum().round(3))

plt.figure(figsize=(7, 5))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_iris, cmap="viridis", alpha=0.7)
plt.title("Iris dataset compressed to 2 dimensions with PCA")
plt.xlabel("PC 1"); plt.ylabel("PC 2")
plt.show()

> 🔭 **Why PCA matters for deep learning.** The intuition that high-dimensional data lives on a lower-dimensional structure is central to deep learning, where networks learn compact internal *representations* of complex inputs. PCA is the classical, linear version of that same idea.

---
## ✏️ Exercises

### Exercise 1
Why must a `StandardScaler` be fitted on the training set only? Describe in one or two sentences what goes wrong otherwise.

In [ ]:
# Your answer here:


<details><summary>💡 Show solution</summary>

```python
# Fitting on the whole dataset lets the test set's mean and standard deviation
# influence the scaling of the training data -- a form of data leakage. The test
# set is no longer truly 'unseen', so reported performance is optimistically biased.
```

*The golden rule: any quantity learned from data (means, mins, vocabularies, etc.) must come from the training set only.*
</details>

### Exercise 2
Use `cross_val_score` to compare a `KNeighborsClassifier(n_neighbors=5)` against a `RandomForestClassifier(n_estimators=100)` on `X_train_scaled` / `y_train` with 5 folds. Report the mean accuracy of each.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
knn = KNeighborsClassifier(n_neighbors=5)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
print('k-NN:', cross_val_score(knn, X_train_scaled, y_train, cv=5).mean().round(3))
print('RF:  ', cross_val_score(rf,  X_train_scaled, y_train, cv=5).mean().round(3))
```
</details>

## 🔑 Key takeaways

- **Scale features** (fit on training data only) so no feature dominates by magnitude alone.
- **Cross-validation** gives a stable, luck-free performance estimate, report mean and spread.
- **Ensembles** like Random Forest combine many weak models into a strong, robust one.
- **Grid search** tunes hyperparameters systematically via cross-validation.
- **k-means** (clustering) and **PCA** (dimensionality reduction) find structure in *unlabelled* data.

### 🏁 You have completed Module C: Machine Learning!
You can frame problems, build and evaluate models, avoid the classic pitfalls, and even work without labels. This classical foundation is exactly what deep learning builds upon.

**Next:** *Module D: Deep Learning*, where we move from a single neuron to modern architectures, starting with Notebook 12.

---
*End of Module C.*

---
*© Ioannis Tsioulis. *From Python to Deep Learning & Fine-Tuning*. Prepared for educational use.*